In [42]:
import pandas as pd
import numpy as np




In [43]:
# EDA
train = pd.read_csv("d:/train.csv")
test = pd.read_csv("d:/test.csv")

full = pd.concat([train,test],axis=0)
y = full.pop('SalePrice')
y.skew()
y = np.log1p(y)  # 对标签的处理，正常在合并前做

In [44]:
# 数据处理

# 缺失值
num_cols = full.select_dtypes(include="number").columns.tolist()
cat_cols = full.select_dtypes(include="str").columns.tolist()


nan = full[num_cols].isna().sum()
nan1 = nan[nan/len(full)>0.3]
for col in num_cols:
    if -0.5 <= full[col].skew() <= 0.5:
        full[col] = full[col].fillna(full[col].mean())
    else:
        full[col] = full[col].fillna(full[col].median())


nan = full[cat_cols].isna().sum()
for col in cat_cols:
    if nan[col]/len(full)>0.3:
        full[col] = full[col].fillna('Missing')
    else:
        full[col] = full[col].fillna(full[col].mode()[0])

In [45]:
# 异常值
for col in num_cols:
    q3 = full[col].quantile(0.75)
    q1 = full[col].quantile(0.25)
    iqr = q3 - q1

    upper = q3 + 1.5*iqr
    lower = q1 - 1.5*iqr

    full[col] = full[col].clip(lower=lower,upper=upper)

In [46]:
# 字符列编码：onehot会返回稀疏矩阵或numpy数组。后续不能按df操作，需要配合pipeline才可以，
# 正常单独使用用get_dummies
x_train = full.iloc[:len(train)]
y_train = y.iloc[:len(train)]
y_test = y.iloc[len(train):]
x_test = full.iloc[len(train):]

from sklearn.preprocessing import OneHotEncoder
import pickle
model = OneHotEncoder(handle_unknown='ignore',sparse_output=False)  # 必须加这个

# 2. 编码（得到二维数组）
encoded = model.fit_transform(x_train[cat_cols])

# 3. 转成 DataFrame
encoded_df = pd.DataFrame(
    encoded,
    columns=model.get_feature_names_out(cat_cols),
)

x_train = x_train.drop(cat_cols, axis=1)
x_train = pd.concat([x_train, encoded_df], axis=1)

with open('../outputs/encoder.pkl','wb') as f:
    pickle.dump(model,f)


In [47]:
# 把处理后的数据放进inputs目录
import os
from pathlib import Path


x_train.to_csv("../inputs/x_train_cleaned.csv", index=False, encoding="utf-8")
